# How recall finds things

`recall()` runs up to three retrieval arms and fuses them. This notebook shows what each arm sees
on its own, how the graph arm picks its seeds, how the fusion weighs the arms and why, and the
knobs you have. It needs no key: `HashEmbedder` stands in for a real embedding model.

The store is a small engineering organisation: two teams, four people, three services, two
handovers, a rotation and a constraint. Enough for a question whose answer is two hops from the
words it uses.

In [1]:
from datetime import datetime, timedelta

from anatid import Anatid, HashEmbedder

DIM = 64
db = Anatid.open(":memory:", tenant=1, embedding_dim=DIM, embedder=HashEmbedder(dim=DIM))
t = datetime(2026, 1, 6, 9)
day = timedelta(days=1)

FACTS = [
    ("Ada leads Kestrel", ["Ada", "Kestrel"], "fact", 0),
    ("Bo is a member of Kestrel", ["Bo", "Kestrel"], "fact", 0),
    ("Cy leads Heron", ["Cy", "Heron"], "fact", 0),
    ("Dee is a member of Heron", ["Dee", "Heron"], "fact", 0),
    ("Kestrel owns the ingest service", ["Kestrel", "ingest service"], "fact", 1),
    ("Heron owns the billing API", ["Heron", "billing API"], "fact", 1),
    ("Heron owns the search indexer", ["Heron", "search indexer"], "fact", 1),
    ("The billing API depends on the ingest service", ["billing API", "ingest service"], "fact", 2),
    ("Bo is on call for Kestrel this month", ["Bo", "Kestrel"], "fact", 30),
    ("Dee is on call for Heron this month", ["Dee", "Heron"], "fact", 30),
    ("The ingest service deploys on Tuesdays between 10:00 and 12:00 UTC", ["ingest service"], "constraint", 3),
    ("The search indexer must stay on OpenSearch 2.11 until the reindex finishes", ["search indexer"], "constraint", 4),
    ("Kestrel holds its planning day on the first Monday of the quarter", ["Kestrel"], "fact", 5),
]
for content, entities, kind, offset in FACTS:
    db.remember(content, entities=entities, kind=kind, writer="notes", now=t + offset * day)
RELATIONS = [
    ("Ada", "Kestrel", "leads"), ("Bo", "Kestrel", "member_of"), ("Cy", "Heron", "leads"),
    ("Dee", "Heron", "member_of"), ("Kestrel", "ingest service", "owns"), ("Heron", "billing API", "owns"),
    ("Heron", "search indexer", "owns"), ("billing API", "ingest service", "depends_on"),
    ("Bo", "Kestrel", "on_call_for"), ("Dee", "Heron", "on_call_for"),
]
for src, dst, kind in RELATIONS:
    db.relate(src, dst, rel_kind=kind, writer="notes", now=t)
print(db.stats()["memories"], "memories,", db.stats()["edges_relates"], "relations")


def show(hits, n=6):
    print(f"arms={hits.arms} seeds={hits.seeds} weights={hits.weights}")
    for h in hits[:n]:
        print(f"  [{h.rank}] {h.score:.4f} {h.content!r:<60} via {'+'.join(h.sources)}")

13 memories, 10 relations


## 1. Each arm on its own

The question asks who is on call for the team that owns the service the billing API depends on:
billing API → ingest service → Kestrel → Bo, three facts that share no keyword with the answer.
Silencing two arms with `arm_weights` shows what the third finds by itself. Only the order matters:
a lone arm's weight scales its scores, not its ranking.

In [2]:
question = "who is on call for the team that owns the service the billing API depends on"

print("text arm alone (BM25 over the memories)")
show(db.recall(question, k=13, arm_weights={"vector": 0, "graph": 0}))
print("\nvector arm alone (cosine over the embeddings)")
show(db.recall(question, k=13, arm_weights={"text": 0, "graph": 0}))
print("\ngraph arm alone (two hops from the entities the question names)")
show(db.recall(question, k=13, arm_weights={"text": 0, "vector": 0}))

text arm alone (BM25 over the memories)
arms=('vector', 'text', 'graph') seeds=('billing API',) weights={'vector': 0.0, 'text': 0.25, 'graph': 0.0}
  [1] 0.0041 'The billing API depends on the ingest service'              via text
  [2] 0.0040 'Heron owns the billing API'                                 via text
  [3] 0.0040 'Bo is on call for Kestrel this month'                       via text
  [4] 0.0039 'Dee is on call for Heron this month'                        via text
  [5] 0.0038 'Kestrel owns the ingest service'                            via text
  [6] 0.0038 'The ingest service deploys on Tuesdays between 10:00 and 12:00 UTC' via text

vector arm alone (cosine over the embeddings)
arms=('vector', 'text', 'graph') seeds=('billing API',) weights={'vector': 1.0, 'text': 0.0, 'graph': 0.0}
  [1] 0.0164 'The billing API depends on the ingest service'              via vector
  [2] 0.0161 'Heron owns the billing API'                                 via vector
  [3] 0.0159 'The sear

## 2. Where the graph arm starts

With `seed_entity="auto"` (the default) the graph arm matches the query's words against the
tenant's entity names, whole words, case-insensitive, longest name first, at most three seeds.
Hyphenated and spaced spellings find each other. Name an entity to expand from exactly that one,
or pass `seed_entity=None` to leave the graph out.

In [3]:
for q in ("what does the billing-api depend on", "Kestrel planning", "who leads the teams"):
    hits = db.recall(q, k=3)
    print(f"{q!r:<45} seeds={hits.seeds} arms={hits.arms}")

print()
show(db.recall("planning day", seed_entity="Heron", k=4))
print()
show(db.recall("planning day", seed_entity=None, k=4))

'what does the billing-api depend on'         seeds=('billing API',) arms=('vector', 'text', 'graph')
'Kestrel planning'                            seeds=('Kestrel',) arms=('vector', 'text', 'graph')
'who leads the teams'                         seeds=() arms=('vector', 'text')

arms=('vector', 'text', 'graph') seeds=('Heron',) weights={'vector': 1.0, 'text': 0.25, 'graph': 0.5}
  [1] 0.0230 'Cy leads Heron'                                             via vector+graph
  [2] 0.0228 'Dee is a member of Heron'                                   via vector+graph
  [3] 0.0227 'Kestrel owns the ingest service'                            via vector+graph
  [4] 0.0226 'Heron owns the billing API'                                 via vector+graph

arms=('vector', 'text') seeds=() weights={'vector': 1.0, 'text': 0.25}
  [1] 0.0205 'Kestrel holds its planning day on the first Monday of the quarter' via vector+text
  [2] 0.0161 'Ada leads Kestrel'                                          via vector


## 3. The fusion, and its weights

Each arm hands the fusion a ranked list, and a memory scores `sum(weight / (60 + rank))` over the
lists it appears in. The weights are not equal. When the vector arm runs it leads (vector 1.0,
graph 0.5, text 0.25); without it the text arm leads (text 1.0, graph 0.5). Before it votes, the
graph arm's neighbourhood is ordered by cosine similarity to the query (or by BM25 without an
embedding), not by recency, so it votes for what the question is about rather than for what was
written last.

Both settings were chosen on the answer-quality benchmark
([`docs/quality.md`](../../docs/quality.md)): with equal votes a BM25 arm over short extracted
sentences outvoted the vector arm often enough that the fusion answered fewer questions than the
vector arm alone. `hits.weights` reports what a call used; `arm_weights=` overrides any of them.

In [4]:
print("default weights")
show(db.recall(question, k=6))
print("\nequal votes, as anatid 0.4.1 fused")
show(db.recall(question, k=6, arm_weights={"text": 1.0, "graph": 1.0}))

default weights
arms=('vector', 'text', 'graph') seeds=('billing API',) weights={'vector': 1.0, 'text': 0.25, 'graph': 0.5}
  [1] 0.0287 'The billing API depends on the ingest service'              via vector+text+graph
  [2] 0.0282 'Heron owns the billing API'                                 via vector+text+graph
  [3] 0.0275 'The search indexer must stay on OpenSearch 2.11 until the reindex finishes' via vector+text+graph
  [4] 0.0271 'Kestrel holds its planning day on the first Monday of the quarter' via vector+text+graph
  [5] 0.0269 'Kestrel owns the ingest service'                            via vector+text+graph
  [6] 0.0266 'Bo is on call for Kestrel this month'                       via vector+text+graph

equal votes, as anatid 0.4.1 fused
arms=('vector', 'text', 'graph') seeds=('billing API',) weights={'vector': 1.0, 'text': 1.0, 'graph': 1.0}
  [1] 0.0492 'The billing API depends on the ingest service'              via vector+text+graph
  [2] 0.0484 'Heron owns the billing A

## 4. Under the hood: the graph arm's order

`anatid.recall.graph_arm` is the raw two-hop expansion, newest first; `rank_graph_candidates`
reorders the same candidates by the query's own signal. The fusion sees the second list.

In [5]:
from anatid.recall import graph_arm, rank_graph_candidates

con = db.connection
kestrel = db.entity_id("Kestrel", create=False)
raw = graph_arm(con, tenant_id=1, seed_entity_id=kestrel, hops=2, topn=20)
ranked = rank_graph_candidates(con, raw, tenant_id=1, embedding=db.embedder.embed_one(question), dim=DIM)
content = {m.memory_id: m.content for m in (db.get(mid) for mid, _ in raw)}
print("newest first:      ", [content[mid] for mid, _ in raw[:4]])
print("ranked by cosine:  ", [content[mid] for mid, _ in ranked[:4]])

newest first:       ['Bo is on call for Kestrel this month', 'Kestrel holds its planning day on the first Monday of the quarter', 'The ingest service deploys on Tuesdays between 10:00 and 12:00 UTC', 'The billing API depends on the ingest service']
ranked by cosine:   ['The billing API depends on the ingest service', 'Heron owns the billing API', 'Kestrel holds its planning day on the first Monday of the quarter', 'Kestrel owns the ingest service']


## 5. The knobs

- `k` is how many hits come back; `candidates` is how deep each arm looks before the fusion.
- `kinds=` keeps only memories of those kinds.
- `as_of=` answers from what the database believed at a time; a superseded fact reappears.
- The text arm is a derived index with a journal: a memory written a moment ago is searchable at
  once. `fts_status()` says how many documents were merged from the journal; `rebuild_fts_index()`
  folds them into a new generation, which buys speed, not visibility.

In [6]:
print("constraints only:", [h.content for h in db.recall("service", k=5, kinds=["constraint"])])

later = t + 60 * day
db.supersede(db.recall("Bo is on call", k=1)[0].memory_id, "Ada is on call for Kestrel this month",
             writer="notes", now=later)
print("now:   ", [h.content for h in db.recall("on call for Kestrel", k=1)])
print("as of: ", [h.content for h in db.recall("on call for Kestrel", k=1, as_of=t + 45 * day)])

status = db.fts_status()
print(f"text index: available={status.available} stale={status.stale} merged from the journal={status.pending_rows}")
db.rebuild_fts_index(now=later)
print("after rebuild: merged from the journal =", db.fts_status().pending_rows)

constraints only: ['The ingest service deploys on Tuesdays between 10:00 and 12:00 UTC', 'The search indexer must stay on OpenSearch 2.11 until the reindex finishes']
now:    ['Ada is on call for Kestrel this month']


as of:  ['Bo is on call for Kestrel this month']
text index: available=True stale=False merged from the journal=14
after rebuild: merged from the journal = 0


## 6. Real embeddings

`OpenAICompatibleEmbedder` speaks to any `/embeddings` endpoint. Open the database with
`embedding_dim` set to that model's width and pass the embedder; every `remember`, `supersede`
and `recall` embeds what it is not given a vector for. The vector arm is a brute-force cosine scan
by default, comfortable to about 100,000 memories per tenant; `anatid.vector.attach()` adds an
HNSW generation beyond that. The cell below runs only with a key.

In [7]:
import os

key = os.environ.get("OPEN_ROUTER_KEY") or os.environ.get("OPENROUTER_API_KEY")
if key:
    from anatid import OpenAICompatibleEmbedder

    embedder = OpenAICompatibleEmbedder("https://openrouter.ai/api/v1", key, "openai/text-embedding-3-small", 1536)
    with Anatid.open(":memory:", tenant=1, embedding_dim=1536, embedder=embedder) as real:
        real.remember("Ada leads Kestrel", entities=["Ada", "Kestrel"])
        print(real.recall("who runs the Kestrel team", k=1).arms)
else:
    print("Set OPEN_ROUTER_KEY to try a real embedding model; HashEmbedder was used above.")
db.close()

Set OPEN_ROUTER_KEY to try a real embedding model; HashEmbedder was used above.
